# 1.导入需要的包

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor,RandomForestClassifier
from sklearn.model_selection import train_test_split,cross_val_predict,cross_val_score
from causalml.inference.meta import BaseTRegressor,BaseRRegressor
from causalml.metrics import auuc_score,plot_gain

# 2.读取数据

In [2]:
data=pd.read_csv(r'data/data_processed.csv')

# 3.特征处理

In [3]:
cat_cols = ["zip_code", "channel"] 
data = pd.get_dummies(
    data,
    columns=cat_cols,    
    prefix=cat_cols,     
    drop_first=True,     
    dtype=int            
)
data.head()

,recency,history,used_discount,used_bogo,is_referral,offer,conversion,zip_code_Surburban,zip_code_Urban,channel_Phone,channel_Web
0,10,4.958921,1,0,0,Buy One Get One,0,1,0,1,0
1,6,5.796301,1,1,1,No Offer,0,0,0,0,1
2,7,5.196561,0,1,1,Buy One Get One,0,1,0,0,1
3,9,6.515942,1,0,1,Discount,0,0,0,0,1
4,2,3.814190,1,0,0,Buy One Get One,0,0,1,0,1


In [4]:
data['offer']=data['offer'].map({'No Offer':0,'Buy One Get One':1,'Discount':2}).astype(int)
data.head()
print(data.columns)

Index(['recency', 'history', 'used_discount', 'used_bogo', 'is_referral',
       'offer', 'conversion', 'zip_code_Surburban', 'zip_code_Urban',
       'channel_Phone', 'channel_Web'],
      dtype='str')


# 4.使用T学习器估计CATE

In [15]:
X=data.drop(columns=['offer','conversion'])
T=data['offer']
y=data['conversion']
X_train,X_test,T_train,T_test,y_train,y_test=train_test_split(X,T,y,test_size=0.2,random_state=42,stratify=T)
model=RandomForestRegressor(n_estimators=200,random_state=42,n_jobs=-1,max_depth=5,min_samples_leaf=10,min_samples_split=20)
t_learner=BaseTRegressor(learner=model,control_name=0)
t_learner.fit(X=X_train,treatment=T_train,y=y_train)
cate=t_learner.predict(X=X_test)
print(t_learner.t_groups)
cate

[1 2]


array([[-0.00895563,  0.05421949],
       [ 0.08248381,  0.08544746],
       [ 0.09560437,  0.04654163],
       ...,
       [ 0.10595372,  0.06305302],
       [ 0.03137713,  0.09623814],
       [ 0.05446084,  0.07138028]], shape=(12800, 2))

In [16]:
test_data=pd.DataFrame({'Buy One Get One':cate[:,0],'Discount':cate[:,1]})
test_data['test_T']=T_test.values
test_data['test_y']=y_test.values
df_bogo_eval = test_data[test_data['test_T'].isin([0, 1])].copy()
df_discount_eval = test_data[test_data['test_T'].isin([0, 2])].copy()
df_discount_eval['test_T']=df_discount_eval['test_T'].map({0:0,2:1})
# 计算Discount策略的AUUC
auuc_discount = auuc_score(
    df_discount_eval,
    outcome_col='test_y',
    treatment_col='test_T',
    treatment_effect_col='Discount'
)
print(f"Discount策略的AUUC:\n{auuc_discount}")
# 计算BOGO策略的AUUC
auuc_bogo = auuc_score(
    df_bogo_eval,
    outcome_col='test_y',
    treatment_col='test_T',
    treatment_effect_col='Buy One Get One'
)
print(f"\nBOGO策略的AUUC:\n{auuc_bogo}")

Discount策略的AUUC:
Buy One Get One    0.539518
dtype: float64

BOGO策略的AUUC:
Discount    0.580107
dtype: float64


两个策略的排序能力都有点低啊，尝试更换模型或者调参，先更换模型试试。

In [29]:
from xgboost import XGBRegressor,XGBClassifier
model1=XGBRegressor(n_estimators=200,random_state=42,n_jobs=-1,max_depth=5,min_child_weight=10,learning_rate=0.1)
t1_learner=BaseTRegressor(learner=model1,control_name=0)
t1_learner.fit(X=X_train,treatment=T_train,y=y_train)
cate1=t1_learner.predict(X=X_test)
test_data1=pd.DataFrame({'Buy One Get One':cate1[:,0],'Discount':cate1[:,1]})
test_data1['test_T']=T_test.values
test_data1['test_y']=y_test.values
df_bogo_eval1 = test_data1[test_data1['test_T'].isin([0, 1])].copy()
df_discount_eval1 = test_data1[test_data1['test_T'].isin([0, 2])].copy()
df_discount_eval1['test_T']=df_discount_eval1['test_T'].map({0:0,2:1})
# 计算Discount策略的AUUC
auuc_discount1 = auuc_score(
    df_discount_eval1,
    outcome_col='test_y',
    treatment_col='test_T',
    treatment_effect_col='Discount'
)
print(f"Discount策略的AUUC:\n{auuc_discount1}")
# 计算BOGO策略的AUUC
auuc_bogo1 = auuc_score(
    df_bogo_eval1,
    outcome_col='test_y',
    treatment_col='test_T',
    treatment_effect_col='Buy One Get One'
)
print(f"\nBOGO策略的AUUC:\n{auuc_bogo1}")


Discount策略的AUUC:
Buy One Get One    0.62143
dtype: float64

BOGO策略的AUUC:
Discount    0.689411
dtype: float64


选择xgboost之后，模型排序能力显著增强，下一步尝试调参

In [28]:
import itertools
import pandas as pd
from xgboost import XGBRegressor
from causalml.inference.meta import BaseTRegressor
from causalml.metrics import auuc_score
param_grid = {
    'max_depth': [5,6,7,8],
    'learning_rate': [0.05,0.08,0.1],
    'min_child_weight': [5,8,10]
}
keys, values = zip(*param_grid.items())
param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
results = []
# 遍历每组参数
for params in param_combinations:
    print(f"测试: {params}")
    model = XGBRegressor(**params, n_estimators=200, random_state=42, n_jobs=-1)
    t_learner = BaseTRegressor(learner=model, control_name=0)
    t_learner.fit(X=X_train, treatment=T_train, y=y_train)
    cate_t = t_learner.predict(X=X_test)
    test_data_t=pd.DataFrame({'Buy One Get One':cate_t[:,0],'Discount':cate_t[:,1]})
    test_data_t['test_T']=T_test.values
    test_data_t['test_y']=y_test.values
    df_bogo_eval_t = test_data_t[test_data_t['test_T'].isin([0, 1])].copy()
    df_discount_eval_t = test_data_t[test_data_t['test_T'].isin([0, 2])].copy()
    df_discount_eval_t['test_T']=df_discount_eval_t['test_T'].map({0:0,2:1})
    # 计算Discount策略的AUUC
    auuc_discount_t = auuc_score(
        df_discount_eval_t,
        outcome_col='test_y',
        treatment_col='test_T',
        treatment_effect_col='Discount'
    ).iloc[0]
    print(f"Discount策略的AUUC:\n{auuc_discount_t}")
    # 计算BOGO策略的AUUC
    auuc_bogo_t = auuc_score(
        df_bogo_eval_t,
        outcome_col='test_y',
        treatment_col='test_T',
        treatment_effect_col='Buy One Get One'
    ).iloc[0]
    combined = 0.4 * auuc_bogo_t + 0.6 * auuc_discount_t
    results.append({
        'params': params,
        'bogo': auuc_bogo_t,
        'discount': auuc_discount_t,
        'combined': combined
    })
    print(f"  BOGO: {auuc_bogo_t}, Discount: {auuc_discount_t}, 综合: {combined:.4f}\n")
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['combined'].idxmax()]
print("=" * 50)
print(f"最佳参数: {best['params']}")
print(f"BOGO: {best['bogo']}, Discount: {best['discount']}, 综合: {best['combined']}")

测试: {'max_depth': 5, 'learning_rate': 0.05, 'min_child_weight': 5}
Discount策略的AUUC:
0.5837815501736686
  BOGO: 0.6467716849318093, Discount: 0.5837815501736686, 综合: 0.6090

测试: {'max_depth': 5, 'learning_rate': 0.05, 'min_child_weight': 8}
Discount策略的AUUC:
0.5827500274828439
  BOGO: 0.6447582536435161, Discount: 0.5827500274828439, 综合: 0.6076

测试: {'max_depth': 5, 'learning_rate': 0.05, 'min_child_weight': 10}
Discount策略的AUUC:
0.5840607111608961
  BOGO: 0.6444791432194598, Discount: 0.5840607111608961, 综合: 0.6082

测试: {'max_depth': 5, 'learning_rate': 0.08, 'min_child_weight': 5}
Discount策略的AUUC:
0.6073098044771054
  BOGO: 0.6755317474651003, Discount: 0.6073098044771054, 综合: 0.6346

测试: {'max_depth': 5, 'learning_rate': 0.08, 'min_child_weight': 8}
Discount策略的AUUC:
0.6078408198290456
  BOGO: 0.6717233684164914, Discount: 0.6078408198290456, 综合: 0.6334

测试: {'max_depth': 5, 'learning_rate': 0.08, 'min_child_weight': 10}
Discount策略的AUUC:
0.6046622175351402
  BOGO: 0.6684389365943144, Di

因为可视化分析中discount策略对于购买率的提升较大，所以赋予其更高的权重，调参后成绩显著提升，BOGO: 0.8834063021586671, Discount: 0.7464913156819074，最佳参数: {'max_depth': 8, 'learning_rate': 0.1, 'min_child_weight': 5}

# 5.使用R学习器估计CATE

In [30]:
r_learner=BaseRRegressor(outcome_learner=XGBRegressor(n_estimators=200,random_state=42,n_jobs=-1,max_depth=5,min_child_weight=10,learning_rate=0.1),
                           propensity_learner=XGBClassifier(n_estimators=100,random_state=42,n_jobs=-1,max_depth=5,min_child_weight=10,learning_rate=0.1),
                           effect_learner=XGBRegressor(n_estimators=100,random_state=42,n_jobs=-1,max_depth=5,min_child_weight=10,learning_rate=0.1),
                           control_name=0)
r_learner.fit(X=X_train,treatment=T_train,y=y_train)
r_cate=r_learner.predict(X=X_test)
r_cate[:10]


array([[ 0.00954917,  0.061872  ],
       [ 0.05641413,  0.04770279],
       [ 0.0354239 ,  0.02603166],
       [-0.01822136,  0.0345987 ],
       [ 0.06950838,  0.10081063],
       [ 0.03435082,  0.05386053],
       [ 0.05472751,  0.09261114],
       [ 0.09686885,  0.14152046],
       [ 0.03586636,  0.05715981],
       [ 0.08045699,  0.06724163]])

In [32]:
test_data_r=pd.DataFrame({'Buy One Get One':r_cate[:,0],'Discount':r_cate[:,1]})
test_data_r['test_t']=T_test.values
test_data_r['test_y']=y_test.values
df_bogo_eval_r = test_data_r[test_data_r['test_t'].isin([0, 1])].copy()
df_discount_eval_r = test_data_r[test_data_r['test_t'].isin([0, 2])].copy()
df_discount_eval_r['test_t']=df_discount_eval_r['test_t'].map({0:0,2:1})
# 计算Discount策略的AUUC
auuc_discount_r =auuc_score(
    outcome_col='test_y',
    treatment_col='test_t',
    treatment_effect_col='Discount',
    df=df_discount_eval_r
)
print(f"Discount策略的AUUC:\n{auuc_discount_r}")
# 计算BOGO策略的AUUC
auuc_bogo_r = auuc_score(
    outcome_col='test_y',
    treatment_col='test_t',
    treatment_effect_col='Buy One Get One',
    df=df_bogo_eval_r
)
print(f"BOGO策略的AUUC:\n{auuc_bogo_r}")

Discount策略的AUUC:
Buy One Get One    0.554261
dtype: float64
BOGO策略的AUUC:
Discount    0.591772
dtype: float64


同样参数与选择的情况下，R学习器排序水平不如T学习器，尝试调参

模型调参

In [33]:
param_grid = {
    'max_depth': [5, 6, 7, 8],
    'learning_rate': [0.05, 0.08, 0.1],
    'min_child_weight': [5, 8, 10]
}
keys, values = zip(*param_grid.items())
param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
results = []
for params in param_combinations:
    print(f"测试: {params}")
    model = XGBRegressor(**params, n_estimators=100, random_state=42, n_jobs=-1)
    r_learner = BaseRRegressor(
    outcome_learner=model,
    propensity_learner=model,
    effect_learner=model,
    control_name=0
    )
    r_learner.fit(X=X_train, treatment=T_train, y=y_train)
    cate_r = r_learner.predict(X=X_test)
    test_data_r = pd.DataFrame({'Buy One Get One': cate_r[:, 0], 'Discount': cate_r[:, 1]})
    test_data_r['test_T'] = T_test.values
    test_data_r['test_y'] = y_test.values
    df_bogo_eval_r = test_data_r[test_data_r['test_T'].isin([0, 1])].copy()
    df_discount_eval_r = test_data_r[test_data_r['test_T'].isin([0, 2])].copy()
    df_discount_eval_r['test_T'] = df_discount_eval_r['test_T'].map({0: 0, 2: 1})
    auuc_bogo_r = auuc_score(df_bogo_eval_r, outcome_col='test_y', treatment_col='test_T', treatment_effect_col='Buy One Get One').iloc[0]
    auuc_discount_r = auuc_score(df_discount_eval_r, outcome_col='test_y', treatment_col='test_T', treatment_effect_col='Discount').iloc[0]
    combined = 0.4 * auuc_bogo_r + 0.6 * auuc_discount_r
    results.append({
        'params': params,
        'bogo': auuc_bogo_r,
        'discount': auuc_discount_r,
        'combined': combined
    })
    print(f"  BOGO: {auuc_bogo_r:.4f}, Discount: {auuc_discount_r:.4f}, 综合: {combined:.4f}\n")
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['combined'].idxmax()]
print("=" * 50)
print(f"最佳参数: {best['params']}")
print(f"BOGO: {best['bogo']:.4f}, Discount: {best['discount']:.4f}, 综合: {best['combined']:.4f}")

测试: {'max_depth': 5, 'learning_rate': 0.05, 'min_child_weight': 5}
  BOGO: 0.5650, Discount: 0.5336, 综合: 0.5462

测试: {'max_depth': 5, 'learning_rate': 0.05, 'min_child_weight': 8}
  BOGO: 0.5680, Discount: 0.5356, 综合: 0.5485

测试: {'max_depth': 5, 'learning_rate': 0.05, 'min_child_weight': 10}
  BOGO: 0.5705, Discount: 0.5343, 综合: 0.5488

测试: {'max_depth': 5, 'learning_rate': 0.08, 'min_child_weight': 5}
  BOGO: 0.5798, Discount: 0.5464, 综合: 0.5597

测试: {'max_depth': 5, 'learning_rate': 0.08, 'min_child_weight': 8}
  BOGO: 0.5860, Discount: 0.5480, 综合: 0.5632

测试: {'max_depth': 5, 'learning_rate': 0.08, 'min_child_weight': 10}
  BOGO: 0.5899, Discount: 0.5479, 综合: 0.5647

测试: {'max_depth': 5, 'learning_rate': 0.1, 'min_child_weight': 5}
  BOGO: 0.6006, Discount: 0.5565, 综合: 0.5742

测试: {'max_depth': 5, 'learning_rate': 0.1, 'min_child_weight': 8}
  BOGO: 0.5981, Discount: 0.5549, 综合: 0.5722

测试: {'max_depth': 5, 'learning_rate': 0.1, 'min_child_weight': 10}
  BOGO: 0.5971, Discount: 0.5

R学习器即使是调参后，水平也远不如T学习器，所以选择T学习器作为最终模型。

# 6.最终模型

In [34]:
import joblib
final_model = XGBRegressor(
    max_depth=8, 
    learning_rate=0.1, 
    min_child_weight=5, 
    n_estimators=200, 
    random_state=42, 
    n_jobs=-1
)
final_t_learner = BaseTRegressor(learner=final_model, control_name=0)
final_t_learner.fit(X=X, treatment=T, y=y)
joblib.dump(final_t_learner, 't_learner_model.pkl')
print("模型已保存为 t_learner_model.pkl")

模型已保存为 t_learner_model.pkl
